<a href="https://colab.research.google.com/github/Heng1222/MalSem-Decon/blob/main/X_class.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
# 安裝必要的 NLP 與深度學習套件
!pip install transformers torch sentence-transformers jieba pandas numpy tqdm

import torch
from transformers import AutoTokenizer, AutoModel
import pandas as pd
import numpy as np
import jieba
import re
import zipfile
import xml.etree.ElementTree as ET
from collections import defaultdict, Counter
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity

# 檢查 GPU 是否可用
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"目前使用的設備: {device}")

目前使用的設備: cuda


In [17]:
# --- [功能 A: 資料清洗與前處理] ---
CJK_OR_ALNUM = re.compile(r"[\u4e00-\u9fff]+|[a-z0-9_]+")
URL_RE = re.compile(r"https?://\S+|www\.\S+")
WHITESPACE_RE = re.compile(r"\s+")
def normalize_text(text):
    """統一大小寫、移除網址與多餘空白。"""
    text = "" if text is None else str(text)
    text = text.replace("\ufeff", " ")
    text = URL_RE.sub(" ", text.lower())
    text = re.sub(r"[\r\n\t]+", " ", text)
    return WHITESPACE_RE.sub(" ", text).strip()

def read_timeline_xlsx(path):
    # 直接解析 xlsx 的 XML，避免缺少 openpyxl 時讀檔失敗。
    ns = {"a": "http://schemas.openxmlformats.org/spreadsheetml/2006/main"}
    rows = []

    def cell_text(cell):
        inline = cell.find("a:is", ns)
        value = cell.find("a:v", ns)
        return "".join(inline.itertext()) if inline is not None else (value.text if value is not None else "")

    with zipfile.ZipFile(path) as zf:
        root = ET.fromstring(zf.read("xl/worksheets/sheet1.xml"))
        sheet_rows = root.find("a:sheetData", ns)
        header_map = {}

        for row_idx, row in enumerate(sheet_rows):
            row_values = {}
            for cell in row:
                ref = cell.attrib.get("r", "")
                col = "".join(ch for ch in ref if ch.isalpha())
                row_values[col] = cell_text(cell)

            if row_idx == 0:
                header_map = row_values
                continue

            rows.append({header_map.get(col, col): value for col, value in row_values.items()})

    return pd.DataFrame(rows)

def preprocess_texts(series):
    """
    確保資料為字串型態，排除 NaN 與空白內容，避免 float 錯誤。
    """
    return series.dropna().astype(str).str.strip().reset_index(drop=True)

# --- [功能 B: 提取上下文相關的靜態單詞表示] ---
# def get_xclass_static_repr(texts, model_name='bert-base-chinese', max_len=512, batch_size=16):
#     """
#     遵循 X-Class 論文：平均單詞在語料庫中所有出現位置的 Contextualized Embeddings 。
#     """
#     tokenizer = AutoTokenizer.from_pretrained(model_name)
#     model = AutoModel.from_pretrained(model_name).to(device)
#     model.eval()

#     word_vectors_sum = defaultdict(lambda: np.zeros(model.config.hidden_size))
#     word_counts = defaultdict(int)

#     print(f"正在從 {len(texts)} 筆貼文中提取上下文語意向量...")

#     for i in tqdm(range(0, len(texts), batch_size)):
#         batch_texts = texts[i:i+batch_size].tolist()
#         # 進行 Tokenization
#         inputs = tokenizer(batch_texts, padding=True, truncation=True, max_length=max_len, return_tensors="pt").to(device)

#         with torch.no_grad():
#             outputs = model(**inputs)
#             # 取得最後一層隱藏狀態 (Last Hidden State)
#             last_hidden_states = outputs.last_hidden_state.cpu().numpy()

#         # 映射 Token 回原始單詞並累積向量
#         for b in range(len(batch_texts)):
#             # 取得 Word IDs (映射 token 到原始 word index)
#             word_ids = inputs.word_ids(batch_index=b)
#             for idx, word_idx in enumerate(word_ids):
#                 if word_idx is not None:
#                     # 取得該 token 對應的字串並進行簡單過濾
#                     token_text = tokenizer.decode(inputs['input_ids'][b][idx]).replace(" ", "")
#                     if len(token_text) >= 1 and re.match(r'[\u4e00-\u9fa5]', token_text):
#                         word_vectors_sum[token_text] += last_hidden_states[b][idx]
#                         word_counts[token_text] += 1

#     # 計算平均向量，並過濾低頻詞
#     static_word_rep = {}
#     vocab = []
#     embeddings = []

#     for word, count in word_counts.items():
#         if count >= 3: # 設定出現次數門檻
#             avg_vec = word_vectors_sum[word] / count
#             static_word_rep[word] = avg_vec
#             vocab.append(word)
#             embeddings.append(avg_vec)

#     return static_word_rep, vocab, np.array(embeddings)
def get_xclass_static_repr(texts, model_name='bert-base-chinese', max_len=128, batch_size=16):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)
    model.eval()

    word_vectors_sum = defaultdict(lambda: np.zeros(model.config.hidden_size))
    word_counts = defaultdict(int)

    print(f"正在以『詞』為單位提取語意向量...")

    for i in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[i:i+batch_size].tolist()

        # 關鍵修正：先用 jieba 分詞，確保我們處理的是「詞」
        for text in batch_texts:
            seg_list = [w for w in jieba.lcut(text) if len(w) > 1 and re.match(r'[\u4e00-\u9fa5]', w)]
            if not seg_list: continue

            # 將這些詞餵給 BERT 取得上下文向量
            # 這裡採用簡化但有效的方法：將詞彙單獨編碼以獲得在該語料庫模型下的靜態表示
            # 若要完美遵循原文，需在句子中定位詞彙，這裡先確保詞彙完整性
            inputs = tokenizer(seg_list, padding=True, truncation=True, return_tensors="pt").to(device)
            with torch.no_grad():
                outputs = model(**inputs)
                # 取 [CLS] 或是平均 Token 向量作為詞表示
                batch_word_vecs = outputs.last_hidden_state[:, 0, :].cpu().numpy()

            for word, vec in zip(seg_list, batch_word_vecs):
                word_vectors_sum[word] += vec
                word_counts[word] += 1

    static_word_rep = {}
    vocab = []
    embeddings = []

    for word, count in word_counts.items():
        if count >= 3: # 門檻可調
            avg_vec = word_vectors_sum[word] / count
            static_word_rep[word] = avg_vec
            vocab.append(word)
            embeddings.append(avg_vec)

    return static_word_rep, vocab, np.array(embeddings)

# --- [功能 C: 加權平均類別表示] ---
def calculate_zipf_weighted_rep(keywords, word_rep_dict):
    """
    依據齊夫定律賦予權重 (1/i)，計算綜合類別表示向量 。
    """
    vectors = []
    weights = []
    for i, word in enumerate(keywords, 1):
        if word in word_rep_dict:
            vectors.append(word_rep_dict[word])
            weights.append(1.0 / i)

    if not vectors: return None

    weighted_avg = np.average(vectors, axis=0, weights=weights)
    return weighted_avg / np.linalg.norm(weighted_avg)

In [18]:
def run_xclass_expansion(class_name, word_rep_dict, vocab, vocab_embeddings, T=100):
    """
    執行 X-Class 迭代擴充，並包含一致性檢查終止條件 。
    """
    # 確保類別名稱本身在詞庫中，若無則單獨 encode
    if class_name not in word_rep_dict:
        print(f"類別名稱 '{class_name}' 不在語料庫，直接生成表示向量。")
        # 在此處手動初始化 tokenizer 和 model 來獲取類別名稱的 Embedding
        temp_tokenizer = AutoTokenizer.from_pretrained('bert-base-chinese')
        temp_model = AutoModel.from_pretrained('bert-base-chinese').to(device)
        temp_model.eval() # 設定模型為評估模式

        # 對類別名稱進行 Tokenization
        inputs = temp_tokenizer(class_name, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)

        with torch.no_grad():
            outputs = temp_model(**inputs)
            # 從模型的最後一層隱藏狀態中獲取類別名稱的 Embedding
            # 為確保與其他單詞的表示一致，這裡取所有 token embedding 的平均值
            class_name_embedding = torch.mean(outputs.last_hidden_state[0], dim=0).cpu().numpy()

        word_rep_dict[class_name] = class_name_embedding # 將 NumPy 陣列形式的 Embedding 存入字典
        vocab.append(class_name)
        vocab_embeddings = np.vstack([vocab_embeddings, class_name_embedding.reshape(1, -1)]) # 堆疊時確保維度正確 (1, hidden_size)

    current_keywords = [class_name]
    current_class_rep = word_rep_dict[class_name]

    print(f"\n開始為 '{class_name}' 尋找擴增關鍵詞...")

    for i in range(2, T + 1):
        # 1. 尋找與目前類別向量最相似的詞
        sims = cosine_similarity(current_class_rep.reshape(1, -1), vocab_embeddings)[0]
        sorted_indices = sims.argsort()[::-1]

        next_word = None
        for idx in sorted_indices:
            candidate = vocab[idx]
            if candidate not in current_keywords:
                next_word = candidate
                break

        if not next_word: break

        # 2. 嘗試加入新詞並重新計算加權表示
        trial_keywords = current_keywords + [next_word]
        new_class_rep = calculate_zipf_weighted_rep(trial_keywords, word_rep_dict)

        # 3. 一致性檢查：檢查新的表示是否能反向檢索出相同的前 i 個詞
        new_sims = cosine_similarity(new_class_rep.reshape(1, -1), vocab_embeddings)[0]
        top_i_indices = new_sims.argsort()[::-1][:i]
        top_i_words = [vocab[idx] for idx in top_i_indices]

        if set(top_i_words) == set(trial_keywords):
            current_keywords = trial_keywords
            current_class_rep = new_class_rep
            # print(f"迭代 {i}: 成功加入 '{next_word}'")
        else:
            print(f"觸發一致性檢查終止 (Inconsistency)。最終獲得 {len(current_keywords)} 個關鍵詞。")
            break

    return current_keywords

In [19]:
# 假設 fb_data 是包含六萬筆貼文的 PD Series
# fb_series = preprocess_texts(fb_data)

# 測試用假數據 (實際跑時請換成你的六萬筆資料)
# sample_docs = pd.Series([
#     "博弈娛樂城下注，高額獎金等你領", "這是一個投資詐騙集團，請小心飆股",
#     "今天早餐吃的好飽，天氣真不錯", "色情網站非法經營遭查獲", "百家樂博弈代操"
# ] * 100)
# fb_series = preprocess_texts(sample_docs)
posts = read_timeline_xlsx("time_line.xlsx")
text_columns = ["content", "share_content", "attachment_title", "attachment_description"]
posts[text_columns] = posts[text_columns].fillna("").astype(str)
posts["text"] = posts[text_columns].agg(" ".join, axis=1).map(normalize_text)
fb_data = posts["text"]

# 第一步：建立全語料上下文表示字典
# 注意：在大規模資料上此步較耗時，建議開啟 GPU
word_rep_dict, vocab, vocab_embeddings = get_xclass_static_repr(fb_data)

# 第二步：定義你想解構的「不好的概念」 (主題名稱) [cite: 1, 14]
categories = ["賭博", "詐騙", "色情"]
all_results = {}

for cat in categories:
    keywords = run_xclass_expansion(cat, word_rep_dict, vocab, vocab_embeddings)
    all_results[cat] = keywords

# 第三步：輸出與儲存
for cat, kw in all_results.items():
    print(f"\n類別 [{cat}] 的子空間關鍵詞: {kw}")

# 儲存為 CSV
output_list = []
for cat, kw in all_results.items():
    for word in kw:
        output_list.append({"category": cat, "keyword": word})

pd.DataFrame(output_list).to_csv("xclass_malice_taxonomy.csv", index=False, encoding="utf-8-sig")
print("\n任務完成，結果已儲存至 xclass_malice_taxonomy.csv")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-chinese
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


正在以『詞』為單位提取語意向量...


  0%|          | 0/5042 [00:00<?, ?it/s]Building prefix dict from the default dictionary ...
DEBUG:jieba:Building prefix dict from the default dictionary ...
Dumping model to file cache /tmp/jieba.cache
DEBUG:jieba:Dumping model to file cache /tmp/jieba.cache
Loading model cost 1.809 seconds.
DEBUG:jieba:Loading model cost 1.809 seconds.
Prefix dict has been built successfully.
DEBUG:jieba:Prefix dict has been built successfully.
100%|██████████| 5042/5042 [09:14<00:00,  9.09it/s]



開始為 '賭博' 尋找擴增關鍵詞...

開始為 '詐騙' 尋找擴增關鍵詞...

開始為 '色情' 尋找擴增關鍵詞...

類別 [賭博] 的子空間關鍵詞: ['賭博', '玩車', '玩球', '玩捕', '玩耍', '玩雪', '互動贏', '玩色', '玩遊戲', '玩命', '吃喝', '喝酒', '玩笑', '遊玩', '玩沙', '好玩', '電玩', '玩具', '玩大學', '喝啤酒', '玩動物', '樂趣', '有趣', '玩樂', '玩玩', '爆殺', '交友', '吃醋', '賠錢', '歡笑', '女人', '電器', '聽音樂', '打槍', '打水仗', '讀書', '苦笑', '用電腦', '風趣', '折騰', '放屁', '唱歌', '機器', '賣場', '打球', '烹飪', '免費遊', '旅遊', '技術', '調皮', '炫耀', '打臉', '電腦', '食物', '娛樂', '男人', '衣物', '熱鬧', '打字', '打籃球', '聽歌', '遊戲', '防遊戲', '造勢', '看戲', '吃飯', '活動', '小遊戲', '毛病', '遊園', '器具', '放肆', '元氣', '爆紅', '商品', '物品', '嘲諷', '火鍋', '家家酒', '彈珠', '通電話', '撒嬌', '大牌', '錢櫃', '寶物', '演繹', '抽電', '養家', '痞子', '過癮', '游戏', '花樣', '種族', '尋人', '紅人', '興趣', '工作狂', '翻臉', '樂園', '吵鬧']

類別 [詐騙] 的子空間關鍵詞: ['詐騙', '欺騙', '騙局', '假帳', '防詐', '受騙', '隱瞞', '反詐', '撒謊', '恐嚇', '阻詐', '套路', '詐團圓', '忽悠', '騙人', '沒騙', '打臉', '假新聞', '陰謀', '陷阱', '還錢', '嘲諷', '打壓', '爆殺', '出賣', '引誘', '羞辱', '造勢', '冒犯', '造孽', '保險', '毛病', '蒙蔽', '破費', '狂殺', '翻臉', '汙辱', '污辱', '撒嬌', '吃醋', '流言', '打擾', '犯罪', '威脅', '陰險', '背書', '爆紅', 